# **03 Sunderland xG & Match Predictions**

This notebook will look at Sunderland's xG across the season and predict match outcomes.


TODO: In next notebook Using Sunderland's xG to estimate the probability of a win, loss or draw in future matches.

You can however change `TEAM` to whatever team you like.

In [25]:
import pandas as pd
from pathlib import Path

TEAM = "Sunderland" # filter by what team you choose
DATA_FOLDER = Path("data")
CHART_FOLDER = Path("charts")
CHART_FOLDER.mkdir(parents=True, exist_ok=True)

ROLLING_WINDOW = 5

# highest number of goals we'll calculate a probability for
# goals more than this amount are unlikely
MAX_GOALS_TO_SHOW = 6

print("Team:", TEAM)

Team: Sunderland


In [26]:
matches = pd.read_csv(DATA_FOLDER / "understat_team_stats.csv", parse_dates=["date"])

# filter by selected TEAM
team_matches_long = pd.read_csv(DATA_FOLDER / "team_matches_long.csv", parse_dates=["date"])

team_matches = (team_matches_long[team_matches_long["team"] == TEAM].sort_values("date").reset_index(drop=True))

print(team_matches[["date", "opponent", "venue", "xg_for", "xg_against", "goals_for", "goals_against"]].tail())


                  date                 opponent venue    xg_for  xg_against  \
33 2026-04-24 19:00:00        Nottingham Forest  Home  0.614323    1.542080   
34 2026-05-02 15:00:00  Wolverhampton Wanderers  Away  0.923954    1.961310   
35 2026-05-09 15:00:00        Manchester United  Home  1.237600    0.535981   
36 2026-05-17 15:00:00                  Everton  Away  1.283830    1.321810   
37 2026-05-24 15:00:00                  Chelsea  Home  2.331090    0.920885   

    goals_for  goals_against  
33          0              5  
34          1              1  
35          0              0  
36          3              1  
37          2              1  


We now have a DataFrame containing one row for each Sunderland match, including opponent, venue, xG for and against, and actual goals scored and conceded.




# Simulating Match Outcomes

We will use Sunderland's xG data from last season to estimate expected goals and simulate possible match scorelines. These simulations will help use estimate probabilities of Sunderland winning, drawing or losing.

## How the simulation works

The idea is to turn last season's xG into a number of **expected goals** for each 2026-27 fixture, then play every match thousands of times with random goals drawn from those expectations.

1. **Attack and defence strength**: for every team, compare xG created and xG conceded per game against the league average. An attack strength of 1.2 means 20% more xG than the average team, and a defence strength of 1.2 means 20% *more* xG conceded than average (so lower is better defensively).
2. **Expected goals for a fixture**: `league average x home/away factor x attack of one team x defence of the other`.
3. **Simulate goals**: goals in football are well modelled by a Poisson distribution, so we draw a random number of goals for each side from their expected goals.
4. **Repeat the whole season** thousands of times and count wins, draws, losses and points.

**IMPORTANT**
- Ipswich Town, Coventry City and Hull City were not in the Premier League last season, so they have no xG data. So we give them teh average strength of last season's three weakest teams.
- Last season's numbers are the only input. Summer signings, managers and the Europa League workload are not captured.
- Strengths are pulled slightly towards the league average (`SHRINKAGE`), because we are using one season sample.

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp, factorial

N_SIMULATIONS = 10_000    # how many times we play the full season
RANDOM_SEED = 42          # makes the results repeatable
SHRINKAGE = 0.2           # 0 = trust last season fully, 1 = everyone is average
PROMOTED_TEAMS = ["Ipswich Town", "Coventry City", "Hull City"]
POINTS_LAST_SEASON = 54
SAFETY_POINTS = 38        # rough points total that usually keeps a team up

## Sunderland's 2026-27 fixtures

Each row is one Premier League match. Some dates are provisional and subject to change due to European games or high profile derbys, but this won't affect results.

In [28]:
fixtures = pd.read_csv(DATA_FOLDER / "sunderland_fixtures_2026_27.csv")

In [31]:
# sanity checks: 38 games, 19 home and 19 away, every opponent played home and away
print("Fixtures:", len(fixtures))
print(fixtures["venue"].value_counts().to_dict())
print("Opponents not played exactly twice:",
      fixtures["opponent"].value_counts()[lambda counts: counts != 2].to_dict())

# which opponents have no data from last season
teams_in_data = set(team_matches_long["team"].unique())
missing_teams = sorted(set(fixtures["opponent"]) - teams_in_data)
print("Opponents with no 2025-26 data:", missing_teams)
print("Promoted teams: ", sorted(PROMOTED_TEAMS))

Fixtures: 38
{'Away': 19, 'Home': 19}
Opponents not played exactly twice: {}
Opponents with no 2025-26 data: ['Coventry City', 'Hull City', 'Ipswich Town']
Promoted teams:  ['Coventry City', 'Hull City', 'Ipswich Town']
